In [38]:
import pandas as pd
import numpy as np
import re

In [39]:
emb = np.load('./filtered_outputs/text_embeddings.npy')
scores = np.load('./filtered_outputs/semantic_scores.npy')
df = pd.read_csv('./filtered_outputs/semantic_scores_added.csv')
print('embeddings shape:', emb.shape)        # expect (N_docs, 768)
print('scores shape    :', scores.shape)     # expect (N_docs,)
print('csv rows        :', len(df))          # must match the two above
print('score range     :', scores.min(), scores.max(), 'mean=', scores.mean())
print('non-zero rows   :', (emb.any(axis=1)).sum(), '/', len(emb))
print('NaNs in scores  :', np.isnan(scores).sum())
print('semantic_score col matches scores.npy:', np.allclose(df['semantic_score'].values, scores))

/var/folders/02/c1hvrmj11kx0z457p84l6pbc0000gn/T/ipykernel_56671/939633985.py:3: DtypeWarning: Columns (0,2,8,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./filtered_outputs/semantic_scores_added.csv')


embeddings shape: (205861, 768)
scores shape    : (205861,)
csv rows        : 205861
score range     : -0.004185779 0.79062223 mean= 0.3269199
non-zero rows   : 205861 / 205861
NaNs in scores  : 0
semantic_score col matches scores.npy: True


In [40]:
# Seed terms from subject matter experts
CORE_SEED_TERMS = [
    "election integrity",
    "electoral integrity",
    "election fraud",
    "electoral fraud",
    "election crimes",
    "election integrity unit",
    "restoring faith in our elections", # less generic
    "securing the vote",
    "voter fraud",
    "voter integrity",
    "vote integrity",
    "vote accuracy",
    "illegal voting",
    "illegal vote", # picks up potential misses form the above
    "non-citizen voting",
    "non-citizens vote", # picks up potential misses from the above
    "photo id law",
    "voter identification",
    "voter id",
    "voter photo id",
    "photo id requirement",
    "election security",
    "for the people act", # https://en.wikipedia.org/wiki/For_the_People_Act
    "voter suppress",
    "voting rights advancement act",
    "restoring faith in democracy",
    "fair elections",
    "voter suppression",
    "voter discrimination",
]

CONTEXT_SEED_TERMS = [
    "collusion",
    "certification",
    "certify",
    "shelby decision",
    "redistricting",
    "gerrymandering",
    "touchscreens",
    "stop the steal",
    "russian interference",
    "rigged",
    # Below are terms I found by labeling
    "russia investigation",
    "russian meddling",
    "foreign interference",
    "voting rights act",
    "freedom to vote act", # https://www.congress.gov/bill/117th-congress/senate-bill/2747
    "john lewis voting rights",
    "voter empowerment act",
    "electoral count",
    "election interference",
    "election subversion",
    "big lie",
    "2010 election",
    "save act",
]

SECONDARY_TERMS = [
    "mail ballot",
    "absentee ballot",
    "early ballot",
    "vote by mail",
    "mail-in ballot",
    "mail-in voting",
    "hand count",
    "es&s",
    "dominion voting",
    "vote centers",
    "voter registration",
    "ballot security",
    "ballot harvesting",
    "ballot drop box",
    "voting system",
    "voter roll",
    "election worker",
]

# Thresholds
SEMANTIC_THRESHOLD_HIGH = 0.5  # Tier 3
SEMANTIC_THRESHOLD_MED = 0.3  # Tier 2


In [41]:
"""#Tier 1 Filtering (Core Keywords)"""
df = df[df['Party'] != 'Independent'].reset_index(drop=True)

# Create regex pattern
pattern = '|'.join([re.escape(kw) for kw in CORE_SEED_TERMS])

# Filter
mask = df['Body_Subject'].str.contains(pattern, case=False, regex=True, na=False)
tier1_df = df[mask].copy()

tier1_df['tier'] = 1
tier1_df['match_type'] = 'core_keyword'

print(f"\nDocuments matching core keywords: {len(tier1_df):,} ({len(tier1_df)/len(df)*100:.2f}%)")

# Distribution by party
if 'Party' in tier1_df.columns:
    print("\nBy party:")
    party_dist = tier1_df['Party'].value_counts()
    for party, count in party_dist.items():
        print(f"  {party}: {count:,} ({count/len(tier1_df)*100:.1f}%)")

# Distribution by Congress
if 'congress' in tier1_df.columns:
    print("\nBy Congress:")
    congress_dist = tier1_df['congress'].value_counts().sort_index()
    for congress, count in congress_dist.items():
        if pd.notna(congress):
            print(f"  {int(congress)}th: {count:,}")


Documents matching core keywords: 3,451 (1.68%)

By party:
  Republican: 1,754 (50.8%)
  Democrat: 1,697 (49.2%)


In [42]:
"""#Filtering (Context + Semantic)"""

# Context keyword match = general terms (case-insensitive substring)
#   OR the DRE voting-machine acronym, matched CASE-SENSITIVELY with word
#   boundaries so "DRE"/"DREs" hits but "hundred"/"address"/"Andrew" do not.
#   (Plain lowercase "dre" as a substring was matching ~91k docs incidentally.)
context_pattern = '|'.join([re.escape(kw) for kw in CONTEXT_SEED_TERMS + SECONDARY_TERMS])
keyword_mask = (
    df['Body_Subject'].str.contains(context_pattern, case=False, regex=True, na=False)
    | df['Body_Subject'].str.contains(r'\bDREs?\b', regex=True, na=False)   # case-sensitive acronym
)
semantic_mask = df['semantic_score'] > SEMANTIC_THRESHOLD_MED
not_tier1_mask = ~df.index.isin(tier1_df.index)

combined_mask = keyword_mask & semantic_mask & not_tier1_mask
tier2_df = df[combined_mask].copy()

tier2_df['tier'] = 2
tier2_df['match_type'] = 'context_keyword+semantic'

print(f"\nDocuments in Tier 2: {len(tier2_df):,} ({len(tier2_df)/len(df)*100:.2f}%)")

# Distribution by party
if 'Party' in tier2_df.columns:
    print("\nBy party:")
    party_dist = tier2_df['Party'].value_counts()
    for party, count in party_dist.items():
        print(f"  {party}: {count:,} ({count/len(tier2_df)*100:.1f}%)")


Documents in Tier 2: 6,047 (2.94%)

By party:
  Democrat: 3,030 (50.1%)
  Republican: 3,017 (49.9%)


In [43]:
"""Tier 3 Filtering (High Semantic)"""

# Filter: high semantic similarity AND not already captured
semantic_mask = df['semantic_score'] > SEMANTIC_THRESHOLD_HIGH
not_captured_mask = ~df.index.isin(tier1_df.index.union(tier2_df.index))

combined_mask = semantic_mask & not_captured_mask
tier3_df = df[combined_mask].copy()

tier3_df['tier'] = 3
tier3_df['match_type'] = 'semantic_only'

print(f"\nDocuments in Tier 3: {len(tier3_df):,} ({len(tier3_df)/len(df)*100:.2f}%)")

# Show some sample matches from Tier 3 (these are "discoveries")
print("\nSample Tier 3 matches (high semantic similarity, no keyword match):")
print("=" * 80)
sample_tier3 = tier3_df.nlargest(5, 'semantic_score')
for idx, row in sample_tier3.iterrows():
    print(f"\nScore: {row['semantic_score']:.3f}")
    print(f"body Subject: {row['Body_Subject'][:100]}...")
    print("-" * 80)


Documents in Tier 3: 925 (0.45%)

Sample Tier 3 matches (high semantic similarity, no keyword match):

Score: 0.737
body Subject: Do you support requiring proof of citizenship to register to vote in our elections? Dear , Since Jan...
--------------------------------------------------------------------------------

Score: 0.736
body Subject: March 30, 2025 Website About Vern Press Releases Services Resources Legislation Contact Last week, P...
--------------------------------------------------------------------------------

Score: 0.679
body Subject: Social Media Companies Testify on Russian Intereference in 2016 Elections If you are having trouble ...
--------------------------------------------------------------------------------

Score: 0.668
body Subject: We Must Protect the Integrity of the American Vote July 13, 2024 The Scalise Capitol Report We Must ...
--------------------------------------------------------------------------------

Score: 0.641
body Subject: Join me tonight D

In [44]:
"""#Summary Statistics and Visualization"""
# Combine all tiers
all_filtered = pd.concat([tier1_df, tier2_df, tier3_df], ignore_index=True)
total_filtered = len(all_filtered)

print(f"\nOriginal corpus: {len(df):,} documents")
print(f"Total filtered: {total_filtered:,} documents ({total_filtered/len(df)*100:.2f}%)")
print(f"\nBreakdown by tier:")
print(f"  Tier 1 (Core keywords):        {len(tier1_df):,} ({len(tier1_df)/total_filtered*100:.1f}%)")
print(f"  Tier 2 (Context + Semantic):   {len(tier2_df):,} ({len(tier2_df)/total_filtered*100:.1f}%)")
print(f"  Tier 3 (High Semantic):        {len(tier3_df):,} ({len(tier3_df)/total_filtered*100:.1f}%)")

# get distribution  Unix Timestamp in form of date in all_filtered
# cannot convert input 1292951693000.0 with the unit 's', at position 0
all_filtered["date"] = pd.to_datetime(
    pd.to_numeric(all_filtered["Unix Timestamp"], errors="coerce"),
    unit="ms"
).dt.date
print("Min Date:", all_filtered["date"].min())
print("Max Date:", all_filtered["date"].max())


ex = all_filtered.head(5)
all_filtered.info()

duplicate_count = all_filtered.duplicated().sum()
print(f"\nDuplicate documents: {duplicate_count:,}")

all_filtered.drop_duplicates(inplace=True)


Original corpus: 205,847 documents
Total filtered: 10,423 documents (5.06%)

Breakdown by tier:
  Tier 1 (Core keywords):        3,451 (33.1%)
  Tier 2 (Context + Semantic):   6,047 (58.0%)
  Tier 3 (High Semantic):        925 (8.9%)
Min Date: 2010-01-11
Max Date: 2025-09-18
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10423 entries, 0 to 10422
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unix Timestamp  10423 non-null  object 
 1   BioGuide ID     10423 non-null  object 
 2   Congress        10423 non-null  object 
 3   First Name      10423 non-null  object 
 4   Last Name       10423 non-null  object 
 5   Date of Birth   10391 non-null  object 
 6   Gender          10423 non-null  object 
 7   State           10423 non-null  object 
 8   District        9474 non-null   object 
 9   Party           10423 non-null  object 
 10  Chamber         10423 non-null  object 
 11  Nickname        641 non-

In [45]:
# ============================================================
# Corpus-wide semantic_score distribution.
# Shows how rare high scores are: ~98% of docs sit below 0.5, and the tail
# above 0.5 nearly vanishes -- which is why the high score bands can't be
# balanced when sampling (you take all available instead).
# ============================================================
edges  = [0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, np.inf]
labels = ['[0.0, 0.2)', '[0.2, 0.3)', '[0.3, 0.4)', '[0.4, 0.5)',
          '[0.5, 0.6)', '[0.6, 0.7)', '[0.7+)']

binned = pd.cut(df['semantic_score'], bins=edges, labels=labels, right=False)
hist = binned.value_counts().reindex(labels)        # keep band order
total = len(df)

print(f"{'score range':<12} {'count':>9} {'share':>7}")
for band, count in hist.items():
    print(f"{band:<12} {count:>9,} {count/total*100:>6.1f}%")
print(f"{'TOTAL':<12} {total:>9,}")

score range      count   share
[0.0, 0.2)      10,022    4.9%
[0.2, 0.3)      61,370   29.8%
[0.3, 0.4)     102,048   49.6%
[0.4, 0.5)      30,305   14.7%
[0.5, 0.6)       1,712    0.8%
[0.6, 0.7)         331    0.2%
[0.7+)              54    0.0%
TOTAL          205,847


In [46]:
# ============================================================
# Scrub email/web-scraping boilerplate from Body_Subject before saving.
# Words like "pdf", "link", "click" and URL fragments are scraping artifacts,
# not content -- they were surfacing as Word2Vec neighbors in step 4
# (e.g. "office_link_pdf", "clicking_link_pdf"). Removing them here, at the
# single source CSV, scrubs them from EVERY step-3 model at once, so we don't
# have to duplicate a stopword patch in each script's preprocess().
#
# Whole-word, case-insensitive (\b...\b) so we drop the standalone token but
# never a substring ("link" inside "linked", "com" inside "comment"). This list
# mirrors NEIGHBOR_BLOCKLIST in step_4_nearest_neighbors_network.py -- keep
# the two in sync. NOTE: corpus-wide text edit, so any step-3 re-run must happen
# AFTER re-running this notebook for the change to take effect.
# ============================================================
BOILERPLATE_TOKENS = [
    'pdf', 'link', 'links', 'click', 'clicking', 'clicked',
    'http', 'https', 'www', '.com', 'html', 'url', 'href',
    'unsubscribe', 'email', 'attachment', 'attached', 'fill',
]
boiler_pattern = r'\b(?:' + '|'.join(BOILERPLATE_TOKENS) + r')\b'

_before_chars = all_filtered['Body_Subject'].astype(str).str.len().sum()
_hits = all_filtered['Body_Subject'].astype(str).str.contains(
    boiler_pattern, case=False, regex=True, na=False)

all_filtered['Body_Subject'] = (
    all_filtered['Body_Subject'].astype(str)
    .str.replace(boiler_pattern, ' ', case=False, regex=True)  # drop the tokens
    .str.replace(r'\s+', ' ', regex=True)                      # collapse gaps
    .str.strip()
)

_after_chars = all_filtered['Body_Subject'].astype(str).str.len().sum()
print(f"Boilerplate scrub: {_hits.sum():,} / {len(all_filtered):,} docs affected "
      f"({_hits.mean()*100:.1f}%)")
print(f"Characters removed: {_before_chars - _after_chars:,}")

Boilerplate scrub: 6,933 / 10,420 docs affected (66.5%)
Characters removed: 141,101


In [47]:
# save all_filtered as csv to ./filtered_outputs
all_filtered.to_csv("./filtered_outputs/all_filtered_combined.csv", index=False)

In [48]:
print(len(all_filtered), "filtered documents saved to ./filtered_outputs/all_filtered_combined.csv")
# print number emails by each party
print("Number of emails by each party:")
print(all_filtered['Party'].value_counts())
print("Percentage of emails by each party:")
print(all_filtered['Party'].value_counts(normalize=True))

print("Percentage of documents captured from original corpus: {:.2f}%".format(len(all_filtered)/len(df)*100))

10420 filtered documents saved to ./filtered_outputs/all_filtered_combined.csv
Number of emails by each party:
Party
Republican    5464
Democrat      4956
Name: count, dtype: int64
Percentage of emails by each party:
Party
Republican    0.524376
Democrat      0.475624
Name: proportion, dtype: float64
Percentage of documents captured from original corpus: 5.06%
